# Predictive Alerting for Cloud Metrics
## Exploration and Hypothesis Testing

**Author**: Ignacio Garbayo Fernandez  
**License**: MIT  

---

### Problem statement

Given the last **W** timesteps of a cloud metric, predict whether an incident will occur in the next **H** timesteps.

$$X_t = \{x_{t-W}, \ldots, x_{t-1}\} \qquad Y_t = \mathbf{1}[\exists\, \text{incident in } \{y_t, \ldots, y_{t+H-1}\}]$$

This is a **binary classification** problem, not time-series regression.  
We do not predict the exact future value — only whether an incident will occur.

### Methodology: Hypothesis-Driven Development (HDD)

Each section below follows the scientific method:
1. **State the hypothesis** and predicted outcome *before running any code*
2. **Run the experiment**
3. **State the conclusion** (confirmed / rejected / partial)

This structure ensures the notebook is a record of scientific reasoning,
not just a collection of outputs.

In [ ]:
import sys
import os

# Allow imports from src/ when running from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier

from src import AGGRESSIVE, BALANCED, CONSERVATIVE, CANONICAL_THRESHOLDS, AlertPolicy
from src.generate_data import generate_synthetic_data
from src.preprocess import create_sliding_windows, temporal_split
from src.model import AlertPredictor
from src.evaluate import (
    plot_precision_recall_curve,
    plot_threshold_comparison,
    plot_feature_importances,
    threshold_sweep,
    print_classification_report,
)

plt.rcParams['figure.dpi'] = 110
print('Setup complete.')

---
## Section 1 — Data Exploration

First, we generate the synthetic dataset and understand its structure.
Understanding your data before training is not optional — it is the first
check against the assumptions your model will make.

In [ ]:
# Generate (or reload) the synthetic time series
from pathlib import Path

DATA_PATH = Path('../data/synthetic_metrics.csv')
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print('Loaded existing dataset.')
else:
    df = generate_synthetic_data()
    print('Generated new dataset.')

print(f'Shape:         {df.shape}')
print(f'Columns:       {list(df.columns)}')
print(f'Incident rate: {df["is_incident"].mean():.2%}')
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
# Visualise the full time series with incident regions shaded
fig, ax = plt.subplots(figsize=(15, 4))

ax.plot(df['t'], df['metric'], lw=0.6, color='steelblue', label='metric')

# Shade incident windows in red
in_incident = False
for i, row in df.iterrows():
    if row['is_incident'] == 1 and not in_incident:
        start = row['t']
        in_incident = True
    elif row['is_incident'] == 0 and in_incident:
        ax.axvspan(start, row['t'], alpha=0.3, color='red', label='incident' if start == df[df['is_incident']==1]['t'].iloc[0] else '')
        in_incident = False

ax.set_xlabel('Timestep (t)')
ax.set_ylabel('Metric value')
ax.set_title('Synthetic cloud metric — full 10,000-step series')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Zoom in on a single incident to understand the spike shape
first_incident_t = df[df['is_incident'] == 1]['t'].iloc[0]
zoom_start = max(0, first_incident_t - 30)
zoom_end   = min(len(df), first_incident_t + 60)
zoom = df[(df['t'] >= zoom_start) & (df['t'] < zoom_end)]

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(zoom['t'], zoom['metric'], lw=1.5, color='steelblue')
ax.fill_between(zoom['t'], zoom['metric'].min(), zoom['metric'],
                where=zoom['is_incident'] == 1, alpha=0.4, color='red', label='incident')
ax.set_title(f'Zoom: incident window around t={first_incident_t}')
ax.set_xlabel('Timestep (t)')
ax.set_ylabel('Metric value')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Class balance — this is why we do NOT use accuracy
counts = df['is_incident'].value_counts().sort_index()
print('Class distribution:')
print(f'  No incident (0): {counts[0]:,}  ({counts[0]/len(df):.1%})')
print(f'  Incident    (1): {counts[1]:,}  ({counts[1]/len(df):.1%})')
print()
print('A naive model that always predicts 0 would achieve:')
naive_accuracy = counts[0] / len(df)
print(f'  Accuracy: {naive_accuracy:.1%}  ← useless but looks great!')
print(f'  Recall:   0.0%   ← misses every single incident')
print()
print('This is why we use PR-AUC and Recall as primary metrics.')

---
## Hypothesis H1 — Window size sensitivity

### Stated hypothesis (before running any code)

**H1**: A lookback window of W=15 captures sufficient temporal context to
predict incidents. A shorter window (W=5) loses too much context and performs
noticeably worse.

**Prediction**: PR-AUC will be meaningfully higher for W=15 than W=5,
and similar between W=15 and W=30.

**Falsification criterion**: If the difference in PR-AUC between W=5 and W=15
is less than 0.02, we conclude that window size does not matter much for this
dataset — short windows are sufficient.

In [ ]:
H = 5   # fixed horizon for H1 experiment
results_h1 = {}

for W in [5, 15, 30, 60]:
    X, y = create_sliding_windows(df, W=W, H=H)
    X_train, X_test, y_train, y_test = temporal_split(X, y)
    
    predictor = AlertPredictor()
    predictor.train(X_train, y_train)
    y_proba = predictor.predict_proba(X_test)
    
    auc = average_precision_score(y_test, y_proba)
    results_h1[W] = {'auc': auc, 'n_train': len(X_train), 'n_test': len(X_test)}
    print(f'W={W:3d}: PR-AUC = {auc:.4f}  (train={len(X_train)}, test={len(X_test)})')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
Ws = sorted(results_h1.keys())
aucs = [results_h1[w]['auc'] for w in Ws]
ax.plot(Ws, aucs, marker='o', lw=2, color='steelblue')
ax.set_xlabel('Lookback window W (timesteps)')
ax.set_ylabel('PR-AUC on test set')
ax.set_title('H1: PR-AUC vs window size W  (H=5 fixed)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### H1 conclusion

*(Fill in after running the cells above.)*

```
W=5  PR-AUC = ___
W=15 PR-AUC = ___
W=30 PR-AUC = ___
W=60 PR-AUC = ___

Delta (W=15 − W=5) = ___

H1 result: [CONFIRMED / REJECTED / PARTIAL]
Interpretation: ...
```

**Design decision**: We use **W=15** for all subsequent experiments as the
default window size (revisit if H1 is rejected).

---
## Hypothesis H2 — Labeling strategy

### Stated hypothesis (before running any code)

**H2**: Using `any()` over the horizon ("predict if any incident in the next H steps")
produces better early-warning capability than using only the last step of the horizon
("predict if there is an incident exactly at t+H").

**Prediction**: The `any()` labeling strategy will achieve higher recall,
because it labels windows as positive earlier — further in advance of the incident.

**Falsification criterion**: If recall under `any()` is less than 5 percentage
points higher than under `last()`, the labeling strategy does not meaningfully
affect early-warning capability.

In [ ]:
W_h2, H_h2 = 15, 5

metric_vals = df['metric'].values
incidents   = df['is_incident'].values
n = len(df)

# Strategy A: any() — label=1 if ANY of the next H steps is an incident
X_any, y_any = [], []
for t in range(W_h2, n - H_h2):
    X_any.append(metric_vals[t - W_h2 : t])
    y_any.append(1 if np.any(incidents[t : t + H_h2] == 1) else 0)
X_any, y_any = np.array(X_any), np.array(y_any)

# Strategy B: last() — label=1 only if the step exactly at t+H is an incident
X_last, y_last = [], []
for t in range(W_h2, n - H_h2):
    X_last.append(metric_vals[t - W_h2 : t])
    y_last.append(int(incidents[t + H_h2 - 1]))
X_last, y_last = np.array(X_last), np.array(y_last)

results_h2 = {}
for label, X, y in [('any() labeling', X_any, y_any), ('last-step labeling', X_last, y_last)]:
    X_train, X_test, y_train, y_test = temporal_split(X, y)
    p = AlertPredictor()
    p.train(X_train, y_train)
    proba = p.predict_proba(X_test)
    preds = (proba >= 0.5).astype(int)
    results_h2[label] = {
        'pr_auc':    average_precision_score(y_test, proba),
        'recall':    recall_score(y_test, preds, zero_division=0),
        'precision': precision_score(y_test, preds, zero_division=0),
        'positive_rate': y_test.mean(),
    }

print(f'{"Strategy":<22} {"PR-AUC":>8} {"Recall":>8} {"Precision":>10} {"Pos rate":>10}')
print('-' * 62)
for name, r in results_h2.items():
    print(f'{name:<22} {r["pr_auc"]:>8.4f} {r["recall"]:>8.4f} {r["precision"]:>10.4f} {r["positive_rate"]:>10.2%}')

### H2 conclusion

*(Fill in after running the cells above.)*

```
any() labeling:       PR-AUC=___, Recall=___
last-step labeling:   PR-AUC=___, Recall=___

Delta recall = ___

H2 result: [CONFIRMED / REJECTED / PARTIAL]
Interpretation: ...
```

**Design decision**: We use **`any()` labeling** for all subsequent experiments
as it better represents the early-warning operational goal.

---
## Hypothesis H3 — Class weighting

### Stated hypothesis (before running any code)

**H3**: Using `class_weight='balanced'` in the Random Forest significantly
improves recall on the minority class (incidents) compared to no class weighting.

**Prediction**: The balanced model will achieve at least 5 percentage points
higher recall than the unbalanced model, with acceptable precision.

**Falsification criterion**: If Δrecall < 5pp, class weighting provides
negligible benefit on this dataset.

In [ ]:
W_h3, H_h3 = 15, 5
X_h3, y_h3 = create_sliding_windows(df, W=W_h3, H=H_h3)
X_train_h3, X_test_h3, y_train_h3, y_test_h3 = temporal_split(X_h3, y_h3)

results_h3 = {}
for label, class_weight in [('balanced', 'balanced'), ('unbalanced (None)', None)]:
    clf = RandomForestClassifier(n_estimators=100, class_weight=class_weight, random_state=42)
    clf.fit(X_train_h3, y_train_h3)
    proba = clf.predict_proba(X_test_h3)[:, 1]
    preds = (proba >= 0.5).astype(int)
    results_h3[label] = {
        'pr_auc':    average_precision_score(y_test_h3, proba),
        'recall':    recall_score(y_test_h3, preds, zero_division=0),
        'precision': precision_score(y_test_h3, preds, zero_division=0),
    }

print(f'{"Class weight":<22} {"PR-AUC":>8} {"Recall":>8} {"Precision":>10}')
print('-' * 52)
for name, r in results_h3.items():
    print(f'{name:<22} {r["pr_auc"]:>8.4f} {r["recall"]:>8.4f} {r["precision"]:>10.4f}')

delta_recall = results_h3['balanced']['recall'] - results_h3['unbalanced (None)']['recall']
print(f'\nΔ recall (balanced − unbalanced) = {delta_recall:+.4f} ({delta_recall*100:+.1f} pp)')

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(7, 4))
names = list(results_h3.keys())
x = np.arange(len(names))
width = 0.3
ax.bar(x - width/2, [results_h3[n]['precision'] for n in names], width, label='Precision', color='steelblue')
ax.bar(x + width/2, [results_h3[n]['recall']    for n in names], width, label='Recall',    color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('H3: class_weight balanced vs None  (threshold=0.5)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### H3 conclusion

*(Fill in after running the cells above.)*

```
balanced:    Recall=___, Precision=___
unbalanced:  Recall=___, Precision=___
Δ recall = ___

H3 result: [CONFIRMED / REJECTED / PARTIAL]
Interpretation: ...
```

**Design decision**: We use `class_weight='balanced'` in `AlertPredictor`
as the default, as it directly encodes the operational priority: missing
an incident is more costly than a false alert.

---
## Section 5 — Threshold Analysis and Business Policy

The model outputs a **probability**, not a binary decision. The threshold
is a business decision, not a technical one:

- **High recall priority** (critical system): lower threshold → catch more incidents, accept more noise
- **High precision priority** (alert-fatigued team): higher threshold → fewer alerts, risk missing some incidents

This section evaluates the three canonical `AlertThreshold` domain objects.

In [ ]:
# Train the final model (W=15, H=5, any() labeling, class_weight=balanced)
W_final, H_final = 15, 5
X_final, y_final = create_sliding_windows(df, W=W_final, H=H_final)
X_train_f, X_test_f, y_train_f, y_test_f = temporal_split(X_final, y_final)

final_predictor = AlertPredictor()
final_predictor.train(X_train_f, y_train_f)
y_proba_final = final_predictor.predict_proba(X_test_f)

print(f'Test set size:   {len(X_test_f)} windows')
print(f'Incident rate:   {y_test_f.mean():.2%}')
print(f'Random baseline PR-AUC: {y_test_f.mean():.4f}')
pr_auc_final = average_precision_score(y_test_f, y_proba_final)
print(f'Model PR-AUC:           {pr_auc_final:.4f}')
print(f'Lift over baseline:     {pr_auc_final / y_test_f.mean():.1f}×')

In [ ]:
# Precision-Recall curve
fig = plot_precision_recall_curve(y_test_f, y_proba_final, title='Final model — Precision-Recall curve')
plt.show()

In [ ]:
# Threshold sweep across the three canonical business policies
sweep = threshold_sweep(y_test_f, y_proba_final, CANONICAL_THRESHOLDS)

print(f'{"Policy":<16} {"Threshold":>10} {"Precision":>10} {"Recall":>8} {"F1":>8} {"Alerts":>8} {"Incidents":>10}')
print('-' * 76)
for label, r in sweep.items():
    print(f'{label:<16} {r["threshold"]:>10.1f} {r["precision"]:>10.4f} {r["recall"]:>8.4f} {r["f1"]:>8.4f} {r["n_alerts"]:>8} {r["n_incidents"]:>10}')

In [ ]:
# Bar chart comparison
fig = plot_threshold_comparison(y_test_f, y_proba_final, CANONICAL_THRESHOLDS)
plt.show()

In [ ]:
# Full classification report at each threshold
for threshold in CANONICAL_THRESHOLDS:
    print_classification_report(y_test_f, y_proba_final, threshold)
    print()

### Business recommendation

Using the `AlertPolicy` domain model to make the cost trade-off explicit:

```python
# Example: payment processing system
# Missing an incident = SLA breach + customer churn → cost 50×
# False alert = on-call engineer investigates for 15 min → cost 1×
payment_policy = AlertPolicy(
    threshold=AGGRESSIVE,
    missed_incident_cost=50.0,
    false_alert_cost=1.0,
)
# cost_ratio = 50 → strongly prefer AGGRESSIVE (high recall)

# Example: internal dev tool
# Incidents are annoying but not catastrophic
# Alert fatigue is a serious concern
devtool_policy = AlertPolicy(
    threshold=CONSERVATIVE,
    missed_incident_cost=2.0,
    false_alert_cost=1.0,
)
# cost_ratio = 2 → BALANCED or CONSERVATIVE is fine
```

*(Fill in the actual recommendation based on the sweep results above.)*

In [ ]:
# Feature importances: which window positions matter most?
fig = plot_feature_importances(
    final_predictor.feature_importances,
    W=W_final,
    title='Feature importances by window position (W=15)',
)
plt.show()

importances = final_predictor.feature_importances
most_important_pos = np.argmax(importances)
print(f'Most important position: t-{W_final - most_important_pos}  (index {most_important_pos})')
print(f'Importance value: {importances[most_important_pos]:.4f}')

---
## Summary

| Hypothesis | Result | Key finding |
|-----------|--------|-------------|
| H1: W=15 sufficient | — | Fill in after running |
| H2: any() labeling | — | Fill in after running |
| H3: class_weight balanced | — | Fill in after running |

### Final model configuration

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| W (window) | 15 | Confirmed by H1 |
| H (horizon) | 5 | 5-step early warning |
| Labeling | any() | Confirmed by H2 |
| Model | RandomForest | KISS — tabular data, interpretable |
| class_weight | balanced | Confirmed by H3 |
| Primary metric | PR-AUC | Correct for imbalanced classification |

### Limitations to acknowledge

1. **Synthetic data**: results will not transfer directly to real metrics
2. **Single metric**: real alerting systems use multivariate signals
3. **No streaming**: this is batch inference; production needs incremental windowing
4. **No concept drift**: a deployed model would need periodic retraining